# Phase 1 — Augmented Backbone Screening

**Goal:** compare ResNet50, EfficientNet-B0, and ConvNeXt-Tiny under the same training protocol, with compression-aware augmentation, using 3 random seeds.

**Screening configuration**
- Image size: 192×192
- Training samples/epoch: 8,000
- Maximum epochs: 5
- Seeds: 42, 123, 2024
- Models: ResNet50, EfficientNet-B0, ConvNeXt-Tiny
- Fixed validation subset: 2,000 images (1,000 REAL + 1,000 FAKE)
- Final evaluation: full validation set + test HQ/Q60/Q40
- Mixed precision enabledz
- Full backbone trainable with differential learning rates
- The best backbone is selected automatically for later fine-tuning

**Important:** training augmentation is applied only to the training set. Validation/test images are never randomly augmented.

In [1]:
# ============================================================
# CELL 01 — INSTALL / IMPORTS
# ============================================================

!pip -q install timm

import os
import gc
import json
import math
import time
import copy
import random
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms
from torchvision.transforms import functional as TF

import timm
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# ============================================================
# CELL 02 — CONFIGURATION
# ============================================================

SEEDS = [42, 123, 2024]

MODEL_NAMES = [
    "resnet50",
    "efficientnet_b0",
    "convnext_tiny",
]

IMAGE_SIZE = 192

TRAIN_SAMPLES_PER_EPOCH = 8_000
VAL_SUBSET_SIZE = 2_000
VAL_PER_CLASS = VAL_SUBSET_SIZE // 2

NUM_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2

BATCH_SIZE = 64
NUM_WORKERS = 2

BACKBONE_LR = 1e-4
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4

JPEG_MIN_QUALITY = 40
JPEG_MAX_QUALITY = 100

NUM_CLASSES = 2
CLASS_NAMES = ["REAL", "FAKE"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"

OUTPUT_DIR = Path("/content/phase1_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BEST_DIR = OUTPUT_DIR / "checkpoints"
BEST_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("IMAGE_SIZE:", IMAGE_SIZE)
print("TRAIN_SAMPLES_PER_EPOCH:", TRAIN_SAMPLES_PER_EPOCH)
print("NUM_EPOCHS:", NUM_EPOCHS)
print("SEEDS:", SEEDS)

DEVICE: cuda
IMAGE_SIZE: 192
TRAIN_SAMPLES_PER_EPOCH: 8000
NUM_EPOCHS: 5
SEEDS: [42, 123, 2024]


In [3]:
# ============================================================
# CELL 03 — MOUNT GOOGLE DRIVE
# ============================================================

from google.colab import drive

drive.mount("/content/drive", force_remount=True)

print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


In [4]:
# ============================================================
# CELL 04 — SET DATASET PATH
# ============================================================

# CHANGE THIS if your ZIP/folder is elsewhere.
# Example:
# ZIP_PATH = Path("/content/drive/MyDrive/face_frames.zip")

ZIP_PATH = Path("/content/drive/MyDrive/face_frames.zip")

EXTRACT_ROOT = Path("/content/face_frames")

if ZIP_PATH.exists():
    print("ZIP found:", ZIP_PATH)
else:
    print("ZIP not found at:", ZIP_PATH)
    print("Use the next cell to search your Drive.")

ZIP found: /content/drive/MyDrive/face_frames.zip


In [5]:
# ============================================================
# CELL 05 — OPTIONAL: FIND ZIP IN GOOGLE DRIVE
# ============================================================

matches = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for f in files:
        if f.lower().endswith(".zip") and "face" in f.lower():
            matches.append(os.path.join(root, f))

print("Possible ZIP files:")
for m in matches[:20]:
    print(m)

# If your ZIP is different, set ZIP_PATH manually in Cell 04.

Possible ZIP files:
/content/drive/MyDrive/face_frames.zip


In [6]:
# ============================================================
# CELL 06 — EXTRACT DATASET
# ============================================================

import zipfile

if ZIP_PATH.exists():
    if not EXTRACT_ROOT.exists():
        print("Extracting:", ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall("/content")
        print("Extraction complete.")
    else:
        print("Extraction directory already exists.")
else:
    raise FileNotFoundError(
        f"Could not find {ZIP_PATH}. Set ZIP_PATH in Cell 04."
    )

# Detect the actual dataset root.
candidate_roots = [
    EXTRACT_ROOT,
    Path("/content/face_frames"),
]

DATA_ROOT = None

for p in candidate_roots:
    if (p / "train").exists() and (p / "val").exists() and (p / "test").exists():
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    # Search one level below /content in case ZIP created a wrapper folder.
    for p in Path("/content").iterdir():
        if p.is_dir() and (p / "train").exists() and (p / "val").exists() and (p / "test").exists():
            DATA_ROOT = p
            break

if DATA_ROOT is None:
    raise FileNotFoundError("Could not automatically locate train/val/test.")

print("DATA_ROOT:", DATA_ROOT)

Extracting: /content/drive/MyDrive/face_frames.zip
Extraction complete.
DATA_ROOT: /content/face_frames


In [7]:
# ============================================================
# CELL 07 — DATASET INSPECTION
# ============================================================

def count_images(split_dir):
    result = {}
    for cls in CLASS_NAMES:
        d = split_dir / cls
        if not d.exists():
            raise FileNotFoundError(f"Missing class folder: {d}")
        result[cls] = len([
            p for p in d.rglob("*")
            if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ])
    return result

for split in ["train", "val", "test"]:
    counts = count_images(DATA_ROOT / split)
    print(f"{split:>5}: REAL={counts['REAL']:,} | FAKE={counts['FAKE']:,}")

print("\nExpected structure:")
print(DATA_ROOT / "train/REAL")
print(DATA_ROOT / "train/FAKE")
print(DATA_ROOT / "val/REAL")
print(DATA_ROOT / "val/FAKE")
print(DATA_ROOT / "test/REAL")
print(DATA_ROOT / "test/FAKE")

train: REAL=7,898 | FAKE=46,448
  val: REAL=1,020 | FAKE=5,982
 test: REAL=1,009 | FAKE=5,878

Expected structure:
/content/face_frames/train/REAL
/content/face_frames/train/FAKE
/content/face_frames/val/REAL
/content/face_frames/val/FAKE
/content/face_frames/test/REAL
/content/face_frames/test/FAKE


In [8]:
# ============================================================
# CELL 08 — JPEG / NOISE AUGMENTATIONS
# ============================================================

class RandomJPEGCompression:
    def __init__(self, p=0.5, min_quality=40, max_quality=100):
        self.p = p
        self.min_quality = min_quality
        self.max_quality = max_quality

    def __call__(self, img):
        if random.random() > self.p:
            return img

        quality = random.randint(self.min_quality, self.max_quality)

        if img.mode != "RGB":
            img = img.convert("RGB")

        import io
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=quality)
        buffer.seek(0)

        return Image.open(buffer).convert("RGB")


class RandomGaussianNoise:
    def __init__(self, p=0.20, std=0.015):
        self.p = p
        self.std = std

    def __call__(self, tensor):
        if random.random() > self.p:
            return tensor

        noise = torch.randn_like(tensor) * self.std
        return torch.clamp(tensor + noise, 0.0, 1.0)


# Training augmentation:
# intentionally moderate so that Deepfake facial artifacts are not destroyed.
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
        saturation=0.08,
        hue=0.02
    ),
    RandomJPEGCompression(
        p=0.50,
        min_quality=JPEG_MIN_QUALITY,
        max_quality=JPEG_MAX_QUALITY
    ),
    transforms.RandomApply(
        [transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.8))],
        p=0.20
    ),
    transforms.ToTensor(),
    RandomGaussianNoise(p=0.20, std=0.015),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# Deterministic validation/test preprocessing.
eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print("Training augmentation ready.")
print("Validation/test: deterministic, no random augmentation.")

Training augmentation ready.
Validation/test: deterministic, no random augmentation.


In [9]:
# ============================================================
# CELL 09 — DATASET CLASS
# ============================================================

class FaceFolderDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = Path(root)
        self.transform = transform

        self.samples = []

        for label, cls in enumerate(CLASS_NAMES):
            cls_dir = self.root / cls
            for p in sorted(cls_dir.rglob("*")):
                if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((str(p), label))

        if not self.samples:
            raise RuntimeError(f"No images found in {self.root}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with Image.open(path) as img:
            img = img.convert("RGB")

        if self.transform is not None:
            img = self.transform(img)

        return img, label


train_dataset = FaceFolderDataset(DATA_ROOT / "train", transform=train_transform)
val_dataset = FaceFolderDataset(DATA_ROOT / "val", transform=eval_transform)
test_dataset = FaceFolderDataset(DATA_ROOT / "test", transform=eval_transform)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Train: 54346
Val: 7002
Test: 6887


In [10]:
# ============================================================
# CELL 10 — FIXED BALANCED VALIDATION SUBSET
# ============================================================

def get_class_indices(dataset):
    labels = np.array([label for _, label in dataset.samples])
    return {
        0: np.where(labels == 0)[0].tolist(),
        1: np.where(labels == 1)[0].tolist(),
    }


val_class_indices = get_class_indices(val_dataset)

if len(val_class_indices[0]) < VAL_PER_CLASS or len(val_class_indices[1]) < VAL_PER_CLASS:
    raise ValueError("Validation set does not contain enough images for the requested balanced subset.")

# Fixed subset: identical images for every model and every seed.
rng = np.random.default_rng(2026)

val_real = rng.choice(
    val_class_indices[0],
    size=VAL_PER_CLASS,
    replace=False
)

val_fake = rng.choice(
    val_class_indices[1],
    size=VAL_PER_CLASS,
    replace=False
)

VAL_SUBSET_INDICES = np.concatenate([val_real, val_fake]).tolist()

val_subset = Subset(val_dataset, VAL_SUBSET_INDICES)

print("Fixed validation subset:", len(val_subset))
print("REAL:", VAL_PER_CLASS)
print("FAKE:", VAL_PER_CLASS)

Fixed validation subset: 2000
REAL: 1000
FAKE: 1000


In [11]:
# ============================================================
# CELL 11 — REPRODUCIBILITY + DATALOADERS
# ============================================================

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # Reproducible enough for screening while keeping GPU performance.
    torch.backends.cudnn.benchmark = True


def make_balanced_train_sampler(dataset, seed):
    labels = np.array([label for _, label in dataset.samples])

    class_counts = np.bincount(labels, minlength=NUM_CLASSES)
    class_weights = 1.0 / np.maximum(class_counts, 1)
    sample_weights = class_weights[labels]

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=TRAIN_SAMPLES_PER_EPOCH,
        replacement=True,
        generator=generator,
    )

    return sampler


def make_loader(dataset, sampler=None, shuffle=False):
    kwargs = dict(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )

    if sampler is not None:
        kwargs["sampler"] = sampler
    else:
        kwargs["shuffle"] = shuffle

    return DataLoader(**kwargs)


# Full deterministic validation is used only after training.
full_val_loader = make_loader(val_dataset, shuffle=False)
test_loader = make_loader(test_dataset, shuffle=False)

print("DataLoader configuration ready.")

DataLoader configuration ready.


In [12]:
# ============================================================
# CELL 12 — MODEL FACTORY
# ============================================================

def build_model(model_name):
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=NUM_CLASSES,
    )

    # Full backbone is trainable for architecture screening.
    for p in model.parameters():
        p.requires_grad = True

    return model


def get_parameter_groups(model):
    head_keywords = [
        "fc.",
        "classifier.",
        "head.",
    ]

    head_params = []
    backbone_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        is_head = any(k in name for k in head_keywords)

        if is_head:
            head_params.append(param)
        else:
            backbone_params.append(param)

    return [
        {"params": backbone_params, "lr": BACKBONE_LR},
        {"params": head_params, "lr": HEAD_LR},
    ]


for name in MODEL_NAMES:
    m = build_model(name)
    print(name, "parameters:", sum(p.numel() for p in m.parameters()) / 1e6, "M")
    del m
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

model.safetensors: reconstructing file:   0%|          |  0.00B /  102MB            

model.safetensors: downloading bytes:           |  0.00B            

resnet50 parameters: 23.51213 M


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

efficientnet_b0 parameters: 4.01011 M


model.safetensors: reconstructing file:   0%|          |  0.00B /  114MB            

model.safetensors: downloading bytes:           |  0.00B            

convnext_tiny parameters: 27.821666 M


In [13]:
# ============================================================
# CELL 13 — METRICS
# ============================================================

def compute_metrics(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= 0.5).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
    }

    return metrics


@torch.no_grad()
def evaluate_loader(model, loader):
    model.eval()

    all_labels = []
    all_probs = []

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=USE_AMP
        ):
            logits = model(images)

        probs = torch.softmax(logits, dim=1)[:, 1]

        all_labels.extend(labels.numpy().tolist())
        all_probs.extend(probs.float().cpu().numpy().tolist())

    return compute_metrics(all_labels, all_probs)


def print_metrics(prefix, metrics):
    print(
        f"{prefix} | "
        f"Acc {metrics['accuracy']:.4f} | "
        f"F1 {metrics['f1']:.4f} | "
        f"AUC {metrics['auc']:.4f}"
    )

In [14]:
# ============================================================
# CELL 14 — TRAIN ONE MODEL
# ============================================================

def train_model(model_name, seed):
    seed_everything(seed)

    sampler = make_balanced_train_sampler(train_dataset, seed)
    train_loader = make_loader(train_dataset, sampler=sampler)

    val_loader = make_loader(val_subset, shuffle=False)

    model = build_model(model_name).to(DEVICE)

    if DEVICE.type == "cuda":
        model = model.to(memory_format=torch.channels_last)

    optimizer = torch.optim.AdamW(
        get_parameter_groups(model),
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=NUM_EPOCHS
    )

    criterion = nn.CrossEntropyLoss()

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    best_f1 = -np.inf
    best_state = None
    best_epoch = 0
    patience_counter = 0
    history = []

    checkpoint_path = BEST_DIR / f"{model_name}_seed{seed}.pt"

    print("\n" + "=" * 72)
    print(f"TRAINING: {model_name} | SEED: {seed}")
    print("=" * 72)

    for epoch in range(1, NUM_EPOCHS + 1):
        start = time.time()

        model.train()
        running_loss = 0.0
        n_batches = 0

        for images, labels in train_loader:
            if DEVICE.type == "cuda":
                images = images.to(
                    DEVICE,
                    non_blocking=True,
                    memory_format=torch.channels_last
                )
            else:
                images = images.to(DEVICE, non_blocking=True)

            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=USE_AMP
            ):
                logits = model(images)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item()
            n_batches += 1

        scheduler.step()

        val_metrics = evaluate_loader(model, val_loader)

        epoch_loss = running_loss / max(n_batches, 1)
        elapsed = time.time() - start

        row = {
            "epoch": epoch,
            "loss": epoch_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "time_sec": elapsed,
        }
        history.append(row)

        print(
            f"Epoch [{epoch:02d}/{NUM_EPOCHS}] | "
            f"Loss {epoch_loss:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val AUC {val_metrics['auc']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Time {elapsed:.1f}s"
        )

        if val_metrics["f1"] > best_f1:
            best_f1 = val_metrics["f1"]
            best_epoch = epoch
            patience_counter = 0

            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

            torch.save(
                {
                    "model_name": model_name,
                    "seed": seed,
                    "epoch": epoch,
                    "val_metrics": val_metrics,
                    "model_state_dict": best_state,
                },
                checkpoint_path
            )

        else:
            patience_counter += 1

            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    # Restore best checkpoint in memory.
    if best_state is not None:
        model.load_state_dict(best_state)

    history_df = pd.DataFrame(history)

    print(
        f"Best epoch: {best_epoch} | "
        f"Best Val F1: {best_f1:.4f}"
    )

    return model, history_df, checkpoint_path

In [15]:
# ============================================================
# CELL 15 — PILOT RUN
# ============================================================

# Run this first.
# If this succeeds, run the full 3-model × 3-seed experiment.

pilot_model, pilot_history, pilot_checkpoint = train_model(
    model_name="resnet50",
    seed=42
)

display(pilot_history)

print("Pilot checkpoint:", pilot_checkpoint)


TRAINING: resnet50 | SEED: 42
Epoch [01/5] | Loss 0.6818 | Val F1 0.5759 | Val AUC 0.6295 | Val Acc 0.5965 | Time 70.3s
Epoch [02/5] | Loss 0.6581 | Val F1 0.6571 | Val AUC 0.6689 | Val Acc 0.5935 | Time 60.5s
Epoch [03/5] | Loss 0.6375 | Val F1 0.5708 | Val AUC 0.6883 | Val Acc 0.6270 | Time 60.7s
Epoch [04/5] | Loss 0.6283 | Val F1 0.6466 | Val AUC 0.6949 | Val Acc 0.6360 | Time 60.3s
Early stopping at epoch 4.
Best epoch: 2 | Best Val F1: 0.6571


,epoch,loss,val_accuracy,val_precision,val_recall,val_f1,val_auc,time_sec
0,1,0.681776,0.5965,0.606866,0.548,0.575933,0.629518,70.275880
1,2,0.658085,0.5935,0.568198,0.779,0.657107,0.668949,60.495118
2,3,0.637474,0.6270,0.672087,0.496,0.570771,0.688275,60.741606
3,4,0.628252,0.6360,0.628302,0.666,0.646602,0.694894,60.271743


Pilot checkpoint: /content/phase1_results/checkpoints/resnet50_seed42.pt


In [16]:
# ============================================================
# CELL 16 — FULL PHASE 1 SCREENING
# ============================================================

# After the pilot succeeds, run this cell.
# Total maximum training runs = 3 models × 3 seeds = 9.

all_histories = []
run_records = []

for model_name in MODEL_NAMES:
    for seed in SEEDS:

        # Skip pilot repetition if it already produced a valid checkpoint.
        checkpoint_path = BEST_DIR / f"{model_name}_seed{seed}.pt"

        model, history_df, checkpoint_path = train_model(
            model_name=model_name,
            seed=seed
        )

        history_df["model"] = model_name
        history_df["seed"] = seed
        all_histories.append(history_df)

        run_records.append({
            "model": model_name,
            "seed": seed,
            "checkpoint": str(checkpoint_path),
            "best_val_f1": history_df["val_f1"].max(),
        })

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

histories_df = pd.concat(all_histories, ignore_index=True)
runs_df = pd.DataFrame(run_records)

histories_df.to_csv(OUTPUT_DIR / "training_histories.csv", index=False)
runs_df.to_csv(OUTPUT_DIR / "runs.csv", index=False)

print("Phase 1 training complete.")
display(runs_df)


TRAINING: resnet50 | SEED: 42
Epoch [01/5] | Loss 0.6818 | Val F1 0.5759 | Val AUC 0.6295 | Val Acc 0.5965 | Time 59.9s
Epoch [02/5] | Loss 0.6581 | Val F1 0.6571 | Val AUC 0.6689 | Val Acc 0.5935 | Time 60.1s
Epoch [03/5] | Loss 0.6375 | Val F1 0.5708 | Val AUC 0.6883 | Val Acc 0.6270 | Time 59.9s
Epoch [04/5] | Loss 0.6283 | Val F1 0.6466 | Val AUC 0.6949 | Val Acc 0.6360 | Time 60.1s
Early stopping at epoch 4.
Best epoch: 2 | Best Val F1: 0.6571

TRAINING: resnet50 | SEED: 123
Epoch [01/5] | Loss 0.6817 | Val F1 0.4354 | Val AUC 0.6240 | Val Acc 0.5785 | Time 60.3s
Epoch [02/5] | Loss 0.6640 | Val F1 0.6368 | Val AUC 0.6645 | Val Acc 0.6155 | Time 60.6s
Epoch [03/5] | Loss 0.6423 | Val F1 0.5445 | Val AUC 0.6898 | Val Acc 0.6285 | Time 61.4s
Epoch [04/5] | Loss 0.6313 | Val F1 0.6074 | Val AUC 0.6997 | Val Acc 0.6290 | Time 60.9s
Early stopping at epoch 4.
Best epoch: 2 | Best Val F1: 0.6368

TRAINING: resnet50 | SEED: 2024
Epoch [01/5] | Loss 0.6759 | Val F1 0.4772 | Val AUC 0.628

,model,seed,checkpoint,best_val_f1
0,resnet50,42,/content/phase1_results/checkpoints/resnet50_s...,0.657107
1,resnet50,123,/content/phase1_results/checkpoints/resnet50_s...,0.636750
2,resnet50,2024,/content/phase1_results/checkpoints/resnet50_s...,0.629259
3,efficientnet_b0,42,/content/phase1_results/checkpoints/efficientn...,0.631001
4,efficientnet_b0,123,/content/phase1_results/checkpoints/efficientn...,0.652546
5,efficientnet_b0,2024,/content/phase1_results/checkpoints/efficientn...,0.634816
6,convnext_tiny,42,/content/phase1_results/checkpoints/convnext_t...,0.688283
7,convnext_tiny,123,/content/phase1_results/checkpoints/convnext_t...,0.666667
8,convnext_tiny,2024,/content/phase1_results/checkpoints/convnext_t...,0.675252


In [17]:
# ============================================================
# CELL 17 — LOAD BEST CHECKPOINT
# ============================================================

def load_checkpoint(model_name, seed):
    checkpoint_path = BEST_DIR / f"{model_name}_seed{seed}.pt"

    if not checkpoint_path.exists():
        raise FileNotFoundError(checkpoint_path)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE
    )

    model = build_model(model_name).to(DEVICE)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    return model, checkpoint

In [18]:
# ============================================================
# CELL 18 — CREATE JPEG Q60 / Q40 TEST DATASETS
# ============================================================

class JPEGTestDataset(Dataset):
    def __init__(self, root, quality):
        self.root = Path(root)
        self.quality = quality
        self.samples = []

        for label, cls in enumerate(CLASS_NAMES):
            cls_dir = self.root / cls
            for p in sorted(cls_dir.rglob("*")):
                if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
                    self.samples.append((str(p), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        with Image.open(path) as img:
            img = img.convert("RGB")

        import io
        buffer = io.BytesIO()
        img.save(buffer, format="JPEG", quality=self.quality)
        buffer.seek(0)

        img = Image.open(buffer).convert("RGB")
        img = eval_transform(img)

        return img, label


q60_dataset = JPEGTestDataset(DATA_ROOT / "test", quality=60)
q40_dataset = JPEGTestDataset(DATA_ROOT / "test", quality=40)

q60_loader = make_loader(q60_dataset, shuffle=False)
q40_loader = make_loader(q40_dataset, shuffle=False)

print("HQ test:", len(test_dataset))
print("Q60 test:", len(q60_dataset))
print("Q40 test:", len(q40_dataset))

HQ test: 6887
Q60 test: 6887
Q40 test: 6887


In [19]:
# ============================================================
# CELL 19 — FULL EVALUATION: HQ / Q60 / Q40
# ============================================================

evaluation_rows = []

# Evaluate each seed separately first.
for model_name in MODEL_NAMES:
    for seed in SEEDS:

        model, checkpoint = load_checkpoint(model_name, seed)

        hq = evaluate_loader(model, test_loader)
        q60 = evaluate_loader(model, q60_loader)
        q40 = evaluate_loader(model, q40_loader)

        row = {
            "model": model_name,
            "seed": seed,

            "HQ_accuracy": hq["accuracy"],
            "HQ_precision": hq["precision"],
            "HQ_recall": hq["recall"],
            "HQ_f1": hq["f1"],
            "HQ_auc": hq["auc"],

            "Q60_accuracy": q60["accuracy"],
            "Q60_f1": q60["f1"],
            "Q60_auc": q60["auc"],

            "Q40_accuracy": q40["accuracy"],
            "Q40_f1": q40["f1"],
            "Q40_auc": q40["auc"],

            "Q60_accuracy_drop": hq["accuracy"] - q60["accuracy"],
            "Q40_accuracy_drop": hq["accuracy"] - q40["accuracy"],
        }

        evaluation_rows.append(row)

        del model
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

evaluation_df = pd.DataFrame(evaluation_rows)
evaluation_df.to_csv(
    OUTPUT_DIR / "evaluation_by_seed.csv",
    index=False
)

display(evaluation_df)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [ ]:
# ============================================================
# CELL 20 — MEAN ± STD ACROSS 3 SEEDS
# ============================================================

metric_cols = [
    "HQ_accuracy",
    "HQ_f1",
    "HQ_auc",
    "Q60_accuracy",
    "Q60_f1",
    "Q60_auc",
    "Q40_accuracy",
    "Q40_f1",
    "Q40_auc",
    "Q60_accuracy_drop",
    "Q40_accuracy_drop",
]

summary_rows = []

for model_name in MODEL_NAMES:
    subset = evaluation_df[evaluation_df["model"] == model_name]

    row = {"model": model_name}

    for col in metric_cols:
        row[f"{col}_mean"] = subset[col].mean()
        row[f"{col}_std"] = subset[col].std(ddof=1)

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_df.to_csv(
    OUTPUT_DIR / "phase1_summary_mean_std.csv",
    index=False
)

display(summary_df)

In [ ]:
# ============================================================
# CELL 21 — BACKBONE SELECTION
# ============================================================

# Selection logic:
# 1) Strong clean performance: HQ F1 + HQ AUC
# 2) Strong compression performance: Q40 Accuracy
# 3) Small HQ -> Q40 accuracy degradation
#
# The score is intentionally dominated by compression robustness
# because robustness is a central objective of this project.

selection = summary_df.copy()

def minmax(series, higher_is_better=True):
    s = series.astype(float)
    if s.max() == s.min():
        out = pd.Series(np.ones(len(s)), index=s.index)
    else:
        if higher_is_better:
            out = (s - s.min()) / (s.max() - s.min())
        else:
            out = (s.max() - s) / (s.max() - s.min())
    return out

selection["score_HQ_F1"] = minmax(selection["HQ_f1_mean"], True)
selection["score_HQ_AUC"] = minmax(selection["HQ_auc_mean"], True)
selection["score_Q40_ACC"] = minmax(selection["Q40_accuracy_mean"], True)
selection["score_Q40_DROP"] = minmax(selection["Q40_accuracy_drop_mean"], False)

# Weighted score:
# 25% clean F1
# 20% clean AUC
# 30% Q40 accuracy
# 25% robustness drop
selection["selection_score"] = (
    0.25 * selection["score_HQ_F1"]
    + 0.20 * selection["score_HQ_AUC"]
    + 0.30 * selection["score_Q40_ACC"]
    + 0.25 * selection["score_Q40_DROP"]
)

selection = selection.sort_values(
    "selection_score",
    ascending=False
).reset_index(drop=True)

selection["rank"] = np.arange(1, len(selection) + 1)

selection.to_csv(
    OUTPUT_DIR / "backbone_selection.csv",
    index=False
)

best_backbone = selection.iloc[0]["model"]

print("=" * 72)
print("PHASE 1 — FINAL BACKBONE SELECTION")
print("=" * 72)

display(
    selection[
        [
            "rank",
            "model",
            "HQ_f1_mean",
            "HQ_auc_mean",
            "Q60_accuracy_mean",
            "Q40_accuracy_mean",
            "Q40_accuracy_drop_mean",
            "selection_score",
        ]
    ]
)

print("\nSelected backbone:", best_backbone)

In [ ]:
# ============================================================
# CELL 22 — SAVE SELECTED BACKBONE FOR FINE-TUNING
# ============================================================

# Choose the seed with the best Q40 accuracy among the selected backbone.
selected_rows = evaluation_df[
    evaluation_df["model"] == best_backbone
].copy()

best_seed = selected_rows.sort_values(
    ["Q40_accuracy", "HQ_f1", "HQ_auc"],
    ascending=False
).iloc[0]["seed"]

source_checkpoint = BEST_DIR / f"{best_backbone}_seed{int(best_seed)}.pt"

selected_dir = OUTPUT_DIR / "selected_backbone"
selected_dir.mkdir(parents=True, exist_ok=True)

destination_checkpoint = (
    selected_dir / f"{best_backbone}_best_for_finetuning.pt"
)

shutil.copy2(
    source_checkpoint,
    destination_checkpoint
)

selection_info = {
    "selected_backbone": best_backbone,
    "selected_seed": int(best_seed),
    "checkpoint": str(destination_checkpoint),
    "selection_criterion": {
        "HQ_F1_weight": 0.25,
        "HQ_AUC_weight": 0.20,
        "Q40_accuracy_weight": 0.30,
        "Q40_accuracy_drop_weight": 0.25,
    },
}

with open(selected_dir / "selection_info.json", "w") as f:
    json.dump(selection_info, f, indent=2)

print("Selected backbone:", best_backbone)
print("Selected seed:", int(best_seed))
print("Checkpoint:", destination_checkpoint)

In [ ]:
# ============================================================
# CELL 23 — FINAL REPORT TABLE
# ============================================================

report_table = summary_df.copy()

report_table["HQ F1"] = (
    report_table["HQ_f1_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["HQ_f1_std"].map(lambda x: f"{x:.4f}")
)

report_table["HQ ROC-AUC"] = (
    report_table["HQ_auc_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["HQ_auc_std"].map(lambda x: f"{x:.4f}")
)

report_table["HQ Accuracy"] = (
    report_table["HQ_accuracy_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["HQ_accuracy_std"].map(lambda x: f"{x:.4f}")
)

report_table["Q60 Accuracy"] = (
    report_table["Q60_accuracy_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["Q60_accuracy_std"].map(lambda x: f"{x:.4f}")
)

report_table["Q40 Accuracy"] = (
    report_table["Q40_accuracy_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["Q40_accuracy_std"].map(lambda x: f"{x:.4f}")
)

report_table["Q40 Drop"] = (
    report_table["Q40_accuracy_drop_mean"].map(lambda x: f"{x:.4f}")
    + " ± "
    + report_table["Q40_accuracy_drop_std"].map(lambda x: f"{x:.4f}")
)

final_report = report_table[
    [
        "model",
        "HQ Accuracy",
        "HQ F1",
        "HQ ROC-AUC",
        "Q60 Accuracy",
        "Q40 Accuracy",
        "Q40 Drop",
    ]
].copy()

final_report.to_csv(
    OUTPUT_DIR / "FINAL_PHASE1_REPORT.csv",
    index=False
)

display(final_report)

print("\nSelected backbone:", best_backbone)
print("All Phase 1 outputs:", OUTPUT_DIR)

In [ ]:
# ============================================================
# CELL 24 — OPTIONAL: DOWNLOAD RESULTS AS ZIP
# ============================================================

# Run only after Phase 1 is complete.

RESULT_ZIP = shutil.make_archive(
    "/content/Phase1_results",
    "zip",
    OUTPUT_DIR
)

print("Created:", RESULT_ZIP)